In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import datetime as dt
from functools import partial
import time

def update_num_violation_vs_arrival_time_plot(batch_df, epoch_id, fig, axis):
    data = [row.asDict() for row in batch_df.collect()]
     if not data:
        return
    
    x = [d['window']['start'] for d in data]
    y = [d['count'] for d in data]
    
    axis.clear()
    axis.plot(x, y, 'r-o', label="Violations")
    axis.set_title("Violations vs Arrival Time")
    axis.set_xlabel("Arrival Time")
    axis.set_ylabel("Number of Violations")
    
     # Add interesting points
    if y:
        max_y = max(y)
        min_y = min(y)
        max_idx = y.index(max_y)
        min_idx = y.index(min_y)
        axis.annotate(f"Max: {max_y}", xy=(x[max_idx], max_y), xytext=(x[max_idx], max_y + 1),
                      arrowprops=dict(facecolor='red', arrowstyle="->"))
        axis.annotate(f"Min: {min_y}", xy=(x[min_idx], min_y), xytext=(x[min_idx], min_y - 1),
                      arrowprops=dict(facecolor='green', arrowstyle="->"))
        
    axis.legend()
    fig.tight_layout()
    fig.canvas.draw()
    fig.canvas.flush_events()
    

In [ ]:
def update_speed_vs_arrival_time_plot(batch_df, epoch_id, fig, axis):
    data = [row.asDict() for row in batch_df.collect()]
    if not data:
        return

    x = [row["timestamp"] for row in data]
    y = [row["speed_reading"] for row in data]

    axis.clear()
    axis.plot(x, y, 'b-o', label="Speed")
    axis.set_title("Speed vs Arrival Time")
    axis.set_xlabel("Arrival Time")
    axis.set_ylabel("Speed")

    if y:
        avg_y = sum(y) / len(y)
        max_y = max(y)
        max_idx = y.index(max_y)

        axis.axhline(avg_y, linestyle="--", color="orange", label=f"Avg: {avg_y:.2f}")
        axis.annotate(f"Max: {max_y:.1f}", xy=(x[max_idx], max_y), xytext=(x[max_idx], max_y + 5),
                      arrowprops=dict(facecolor='red', arrowstyle="->"))
    
    axis.legend()
    fig.tight_layout()
    fig.canvas.draw()
    fig.canvas.flush_events()

In [ ]:
num_violation_vs_arrival_time_fig, num_violation_vs_arrival_time_axes = plt.subplots(1, 3, figsize=(15, 4))
num_violation_vs_arrival_time_fig.suptitle("Number of Violations vs Arrival Time")

for axes, title in zip(num_violation_vs_arrival_time_axes, ["Camera Event A", "Camera Event B", "Camera Event C"]):
    axes.set_title(title)
    axes.set_xlabel("Arrival Time")
    axes.set_ylabel("Number of Violations")  

num_violation_vs_arrival_time_fig.show()

In [ ]:
speed_vs_arrival_time_fig, speed_vs_arrival_time_axes = plt.subplots(1, 3, figsize=(15, 4))
speed_vs_arrival_time_fig.suptitle("Speed vs Arrival Time")

for axes, title in zip(speed_vs_arrival_time_axes, ["Camera Event A", "Camera Event B", "Camera Event C"]):
    axes.set_title(title)
    axes.set_xlabel("Arrival Time")
    axes.set_ylabel("Speed")

speed_vs_arrival_time_fig.show()


In [ ]:
last_map_draw = [0]  # use mutable object to persist across calls
MAP_DRAW_INTERVAL = 10  # seconds

def update_plots_combined(batch_df, epoch_id, viol_fig, viol_axis, speed_fig, speed_axis):
    update_num_violation_vs_arrival_time_plot(batch_df, epoch_id, viol_fig, viol_axis)
    update_speed_vs_arrival_time_plot(batch_df, epoch_id, speed_fig, speed_axis)
    
    # === Add map generation logic ===
    current_time = time.time()
    if current_time - last_map_draw[0] > MAP_DRAW_INTERVAL:
        try:
            generate_map()
            print("[Map] Regenerated violation map.")
        except Exception as e:
            print("[Map] Failed to generate map:", e)
        last_map_draw[0] = current_time


In [ ]:
from folium.plugins import MarkerCluster
import folium

def generate_map():
    fmap = folium.Map(location=[-37.81, 144.96], zoom_start=14) # need to change this somewhere
    marker_cluster = MarkerCluster().add_to(fmap)

    # Camera markers
    camera_positions = {}
    for cam in camera_coll.find():
        cam_id = cam["camera_id"]
        lat, lon = cam["lat"], cam["lon"]
        camera_positions[cam_id] = (lat, lon)
        folium.Marker([lat, lon], popup=f"Camera {cam_id}",
                      icon=folium.Icon(color="blue", icon="camera")).add_to(marker_cluster)

    # Violation lines
    pipeline = [
        {"$group": {
            "_id": {"start": "$camera_id_start", "end": "$camera_id_end"},
            "count": {"$sum": 1}
        }}
    ]
    edges = list(violation_coll.aggregate(pipeline))
    for edge in edges:
        start_id, end_id = edge["_id"]["start"], edge["_id"]["end"]
        count = edge["count"]
        if start_id in camera_positions and end_id in camera_positions:
            folium.PolyLine(
                [camera_positions[start_id], camera_positions[end_id]],
                color="red" if count > 5 else "orange",
                weight=3,
                tooltip=f"{start_id} → {end_id}: {count} violations"
            ).add_to(fmap)  # hotspot 

    fmap.save("violation_map.html")


In [2]:
# For camera A
camera_a_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(partial(update_plots_combined,
                          viol_fig=num_violation_vs_arrival_time_fig,
                          viol_axis=num_violation_vs_arrival_time_axes[0],
                          speed_fig=speed_vs_arrival_time_fig,
                          speed_axis=speed_vs_arrival_time_axes[0])) \
    .start()

# For camera B
camera_b_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(partial(update_plots_combined,
                          viol_fig=num_violation_vs_arrival_time_fig,
                          viol_axis=num_violation_vs_arrival_time_axes[1],
                          speed_fig=speed_vs_arrival_time_fig,
                          speed_axis=speed_vs_arrival_time_axes[1])) \
    .start()

# For camera C
camera_c_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(partial(update_plots_combined,
                          viol_fig=num_violation_vs_arrival_time_fig,
                          viol_axis=num_violation_vs_arrival_time_axes[2],
                          speed_fig=speed_vs_arrival_time_fig,
                          speed_axis=speed_vs_arrival_time_axes[2])) \
    .start()




count_violation_camera_stream_a.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

count_violation_camera_stream_b.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

count_violation_camera_stream_c.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

# count_violation_camera_stream_a.writeStream \
#     .outputMode("append") \
#     .foreachBatch(partial(update_num_violation_vs_arrival_time_plot, num_violation_vs_arrival_time_fig, num_violation_vs_arrival_time_axes[0])) \
#     .start()

NameError: name 'camera_a_stream' is not defined